# Security Lookup Example

This example is the lightweight single-security lookup notebook.

Default behavior is safe and fully replayable: it inspects the generated security reference without opening a live TWS connection. Set `RUN_LIVE_LOOKUP = True` only when TWS / IB Gateway is running locally in the `py313` environment.

In [ ]:
from pathlib import Path
import sys
from typing import Dict

project_root = Path.cwd().resolve()
while not (project_root / "market_helper").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root containing the market_helper package.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


def load_local_account_defaults(root: Path) -> Dict[str, str]:
    candidates = [
        root / "configs" / "portfolio_monitor" / "local.env",
    ]
    values: Dict[str, str] = {}
    for path in candidates:
        if not path.exists():
            continue
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip().strip(chr(34)).strip("'")
        break
    return values


def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

local_account_defaults = load_local_account_defaults(project_root)
artifact_dir = project_root / "data" / "artifacts" / "portfolio_monitor"

RUN_LIVE_LOOKUP = False

SYMBOL = "XLK"
SEC_TYPE = "STK"
EXCHANGE = "SMART"
PRIMARY_EXCHANGE = "ARCA"
CURRENCY = "USD"
CONID = None
LOCAL_SYMBOL = None

IB_HOST = "127.0.0.1"
IB_PORT = 7497
IB_CLIENT_ID = 121
IB_ACCOUNT = local_account_defaults.get("DEFAULT_PROD_ACCOUNT_ID", "")

print(f"Project root: {project_root}")
print(f"RUN_LIVE_LOOKUP: {RUN_LIVE_LOOKUP}")
print(f"Lookup target: symbol={SYMBOL}, sec_type={SEC_TYPE}, exchange={EXCHANGE}, primary_exchange={PRIMARY_EXCHANGE}, currency={CURRENCY}, conid={CONID}")
print(f"IBKR connection: host={IB_HOST}, port={IB_PORT}, client_id={IB_CLIENT_ID}, account={IB_ACCOUNT or '<default>'}")


In [ ]:
from pprint import pprint

from IPython.display import display
import pandas as pd

from market_helper.portfolio.security_reference import build_security_reference_table

reference_table = build_security_reference_table()
reference_matches = reference_table.search_by_ibkr_symbol_sec_type(symbol=SYMBOL, sec_type=SEC_TYPE)
reference_frame = pd.DataFrame(
    [
        {
            "internal_id": security.internal_id,
            "canonical_symbol": security.canonical_symbol,
            "display_ticker": security.display_ticker,
            "display_name": security.display_name,
            "asset_class": security.asset_class,
            "primary_exchange": security.primary_exchange,
            "ibkr_exchange": security.ibkr_exchange,
            "ibkr_conid": security.ibkr_conid,
            "yahoo_symbol": security.yahoo_symbol,
            "lookup_status": security.lookup_status,
        }
        for security in reference_matches
    ]
)

print(f"Generated reference matches: {len(reference_matches)}")
display(reference_frame if not reference_frame.empty else pd.DataFrame([{"note": "No generated reference match found."}]))


In [ ]:
live_matches = []

if RUN_LIVE_LOOKUP:
    try:
        from market_helper.providers.tws_ib_async import TwsIbAsyncClient

        client = TwsIbAsyncClient(
            host=IB_HOST,
            port=IB_PORT,
            client_id=IB_CLIENT_ID,
            account=IB_ACCOUNT,
        )
        try:
            client.connect()
            live_matches = client.search_securities(
                symbol=SYMBOL,
                sec_type=SEC_TYPE,
                exchange=EXCHANGE,
                primary_exchange=PRIMARY_EXCHANGE,
                currency=CURRENCY,
                conid=CONID,
                local_symbol=LOCAL_SYMBOL,
            )
        finally:
            client.disconnect()
        print(f"Live IBKR matches: {len(live_matches)}")
        if live_matches:
            display(pd.DataFrame(live_matches).reset_index(drop=True))
        else:
            print("No live matches returned.")
    except Exception as exc:
        print("Live lookup failed cleanly:")
        print(type(exc).__name__, exc)
else:
    print("RUN_LIVE_LOOKUP is False. Set it to True when TWS / IB Gateway is available.")
